# 7.1 · 扩展机制与 Tasks 扩展

**The extension mechanism and the Tasks extension**

> **学完本课你能：** 在双方声明并协商 MCP 扩展，在对端不支持时优雅降级，并实现 Tasks 扩展的“创建—轮询—更新—取消”生命周期。

官方阅读材料：[Extensions overview](https://modelcontextprotocol.io/extensions/overview) · [Extension support matrix](https://modelcontextprotocol.io/extensions/client-matrix) · [Tasks](https://modelcontextprotocol.io/extensions/tasks/overview)

从上到下依次运行单元格（`Shift+Enter`），然后改动代码再重新运行。所有内容都在本地运行：服务器位于当前内核、临时目录或 localhost 上。调用真实大语言模型（LLM）的单元格只有在设置了 `ANTHROPIC_API_KEY` 时才会执行，否则自动跳过。

### 1. 扩展标识符：语法规则
SDK 用与规范相同的规则来校验扩展标识符（Extension Identifier）：以点分隔的标签、必需的 `/`，然后是名称。试一试合法的和不合法的标识符。

In [ ]:
from mcp.shared.extension import validate_extension_identifier

candidates = ["io.modelcontextprotocol/tasks", "com.example/weather-alerts",
              "io.modelcontextprotocol/my-extension-v2",
              "tasks",                 # no vendor prefix
              "example.com/x",         # fine grammatically, but use the REVERSED domain you own
              "com.example/",          # empty name
              "-bad.prefix/thing"]     # label must start with a letter
for ident in candidates:
    try:
        validate_extension_identifier(ident, owner="demo")
        print(f"valid    {ident}")
    except TypeError as exc:
        print(f"INVALID  {ident!r}")

语法规则无法区分 `example.com/x` 和 `com.example/x`。使用<em>反写</em>的域名是文档要求你遵守的约定，而不是解析器能够检查的东西。

### 2. 在双方声明扩展
`MCPServer(extensions=[...])` 接受 `Extension` 子类。每一个都会以其 `settings()` 返回的字典，公布在 `capabilities.extensions[identifier]` 之下。在客户端一侧，`Client(extensions=[...])` 接受 `ClientExtension` 实例，而 `advertise()` 是只带标识符的简化形式。下面的工具读取<strong>本次请求</strong>的客户端能力（Capability），并进行优雅降级（Graceful Degradation）。

In [ ]:
import json

from mcp import Client
from mcp.client.extension import advertise
from mcp.server import MCPServer
from mcp.server.extension import Extension
from mcp.server.mcpserver import Context

ALERTS = "com.example/weather-alerts"


class WeatherAlerts(Extension):
    identifier = ALERTS

    def settings(self):
        return {"version": 1}          # per-extension settings object (schema defined by the extension)


server = MCPServer("weather", extensions=[WeatherAlerts()])


@server.tool()
def forecast(city: str, ctx: Context) -> str:
    """Today's forecast for a city."""
    caps = ctx.client_capabilities
    declared = (caps.extensions or {}) if caps else {}
    text = f"{city}: 18°C, light rain."
    if ALERTS in declared:             # enhanced behaviour only when THIS request declared the extension
        text += " ALERT: wind gusts up to 70 km/h after 18:00."
    return text                        # core fallback is still a meaningful answer


async with Client(server) as plain, Client(server, extensions=[advertise(ALERTS)]) as aware:
    print("server advertises:", json.dumps(plain.server_capabilities.extensions))
    print("plain client ->", (await plain.call_tool("forecast", {"city": "Oslo"})).content[0].text)
    print("aware client ->", (await aware.call_tool("forecast", {"city": "Oslo"})).content[0].text)

### 3. 线路层面：基于原始 stdio 的按请求协商
这里把同样的思路作为一个子进程来运行。客户端能力（Capability）位于每个请求的 `_meta` 中，因此同一条连接上连续的两个 `tools/call` 可以协商出不同的结果。

In [ ]:
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path

workdir = Path(tempfile.mkdtemp())
(workdir / "alerts_server.py").write_text(textwrap.dedent('''
    from mcp.server import MCPServer
    from mcp.server.extension import Extension
    from mcp.server.mcpserver import Context

    class WeatherAlerts(Extension):
        identifier = "com.example/weather-alerts"
        def settings(self):
            return {"version": 1}

    server = MCPServer("weather", extensions=[WeatherAlerts()])

    @server.tool()
    def forecast(city: str, ctx: Context) -> str:
        """Today's forecast for a city."""
        caps = ctx.client_capabilities
        extra = " ALERT: gusts after 18:00." if caps and caps.extensions and "com.example/weather-alerts" in caps.extensions else ""
        return f"{city}: 18C, light rain." + extra

    if __name__ == "__main__":
        server.run(transport="stdio")
'''))

proc = subprocess.Popen([sys.executable, str(workdir / "alerts_server.py")],
                        stdin=subprocess.PIPE, stdout=subprocess.PIPE, text=True)


def meta(extensions=None):
    caps = {"extensions": extensions} if extensions else {}
    return {"io.modelcontextprotocol/protocolVersion": "2026-07-28",
            "io.modelcontextprotocol/clientCapabilities": caps,
            "io.modelcontextprotocol/clientInfo": {"name": "by-hand", "version": "0.1"}}


def rpc(id_, method, params, show=True):
    message = {"jsonrpc": "2.0", "id": id_, "method": method, "params": params}
    proc.stdin.write(json.dumps(message) + "\n")
    proc.stdin.flush()
    reply = json.loads(proc.stdout.readline())
    if show:
        print(">>>", json.dumps(message))
        print("<<<", json.dumps(reply)[:400], "\n")
    return reply


disc = rpc(1, "server/discover", {"_meta": meta()}, show=False)
print("server/discover capabilities.extensions =", disc["result"]["capabilities"].get("extensions"), "\n")
rpc(2, "tools/call", {"name": "forecast", "arguments": {"city": "Oslo"}, "_meta": meta()})
rpc(3, "tools/call", {"name": "forecast", "arguments": {"city": "Oslo"},
                      "_meta": meta({"com.example/weather-alerts": {}})})
proc.stdin.close()
exit_code = proc.wait(timeout=10)

### 4. 强制性扩展：以 `-32021` 拒绝
当不存在有意义的回退方案时，正确的回答是 <em>Missing Required Client Capability</em>（缺少必需的客户端能力）错误。SDK 的辅助函数 `require_client_extension(ctx, identifier)` 会抛出这个错误，并带上 `requiredCapabilities` 载荷。工具处理函数接收的是高层的 `Context`，其 `request_context` 就是该辅助函数所需的底层上下文对象（Context）。

In [ ]:
from mcp.server.mcpserver import require_client_extension
from mcp.shared.exceptions import MCPError

strict = MCPServer("strict", extensions=[WeatherAlerts()])


@strict.tool()
def severe_weather_feed(region: str, ctx: Context) -> str:
    """Stream of severe-weather alerts (only meaningful with the alerts extension)."""
    require_client_extension(ctx.request_context, ALERTS)
    return f"{region}: no active alerts"


async with Client(strict) as plain, Client(strict, extensions=[advertise(ALERTS)]) as aware:
    print("aware ->", (await aware.call_tool("severe_weather_feed", {"region": "NO"})).content[0].text)
    try:
        await plain.call_tool("severe_weather_feed", {"region": "NO"})
    except MCPError as err:
        print("plain -> error", err.code, err.message)
        print("         data:", json.dumps(err.data))

### 5. Tasks 扩展：线路模型
这里没有安装任何 Tasks 软件包，因此我们在 SDK 的扩展 API 之上自己构建这个扩展（Extension），逐字段遵循 `ext-tasks` 规范。首先是各种数据形态：`Task`、`CreateTaskResult`（`resultType: "task"`）、`DetailedTask`（`tasks/get` 的结果，`resultType: "complete"`），以及请求参数。SDK 的模型会把 snake_case 属性转换为线路上的 camelCase 名称。

In [ ]:
import asyncio
import secrets
from datetime import datetime, timezone
from typing import Any, Literal

import mcp_types as types
from mcp_types import Request, RequestParams, Result

TASKS = "io.modelcontextprotocol/tasks"
Status = Literal["working", "input_required", "completed", "failed", "cancelled"]
TERMINAL = {"completed", "failed", "cancelled"}


def now() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="milliseconds").replace("+00:00", "Z")


class Task(Result):
    task_id: str
    status: Status
    status_message: str | None = None
    created_at: str
    last_updated_at: str
    ttl_ms: int | None
    poll_interval_ms: int | None = None


class CreateTaskResult(Task):              # type CreateTaskResult = Result & Task
    result_type: Literal["task"] = "task"


class DetailedTask(Task):                  # type GetTaskResult = Result & DetailedTask
    result_type: Literal["complete"] = "complete"
    input_requests: dict[str, Any] | None = None
    result: dict[str, Any] | None = None
    error: dict[str, Any] | None = None


class TaskIdParams(RequestParams):
    task_id: str


class UpdateTaskParams(TaskIdParams):
    input_responses: dict[str, Any]


def wire(model) -> dict:
    return model.model_dump(by_alias=True, exclude_none=True, mode="json")


print(json.dumps(wire(CreateTaskResult(task_id="t-1", status="working", created_at=now(),
                                       last_updated_at=now(), ttl_ms=60_000, poll_interval_ms=50)), indent=2))

### 6. 服务器端：一个拦截 `tools/call` 的 `Extension`
该扩展使用了 SDK 的两个扩展点。<strong>`intercept_tool_call`</strong> 针对每个请求决定是否返回任务（Task）。<strong>`methods()`</strong> 以附加的 `MethodBinding` 形式提供 `tasks/get`、`tasks/update` 和 `tasks/cancel`。工作以后台 `asyncio` 任务的形式运行。它可以在 `input_required` 状态下暂停并等待 `tasks/update`，并按照规范要求的 `completed` / `failed` 划分来记录终止状态（Terminal Status）。

In [ ]:
from mcp.server.extension import MethodBinding
from mcp_types import INTERNAL_ERROR, INVALID_PARAMS


class TasksExtension(Extension):
    identifier = TASKS

    def __init__(self, task_tools: dict[str, dict], ttl_ms: int = 60_000, poll_interval_ms: int = 50):
        self.task_tools = task_tools     # tool name -> {"needs_approval": bool, "requires_tasks": bool}
        self.ttl_ms, self.poll_interval_ms = ttl_ms, poll_interval_ms
        self.store: dict[str, dict] = {} # stands in for a durable store (database, Redis, ...)
        self.server: MCPServer | None = None

    # ---- helpers -------------------------------------------------------------------------------
    def _set(self, rec, **changes):
        rec.update(changes, last_updated_at=now())

    def _view(self, rec, model=DetailedTask):
        fields = {k: rec.get(k) for k in ("task_id", "status", "status_message", "created_at",
                                          "last_updated_at", "ttl_ms", "poll_interval_ms")}
        if model is DetailedTask:
            if rec["status"] == "input_required":
                fields["input_requests"] = {k: v for k, v in rec["requests"].items() if k not in rec["answers"]}
            fields["result"], fields["error"] = rec.get("result"), rec.get("error")
        return model(**fields)

    def _lookup(self, ctx, task_id):
        require_client_extension(ctx, TASKS)                    # -32021 for non-declaring clients
        rec = self.store.get(task_id)
        if rec is None:
            raise MCPError(INVALID_PARAMS, "Failed to retrieve task: Task not found")
        return rec

    async def _ask(self, rec, key, params):
        """Pause in input_required until tasks/update answers `key` (keys are never reused)."""
        rec["requests"][key] = {"method": "elicitation/create", "params": params}
        rec["waiters"][key] = asyncio.get_running_loop().create_future()
        self._set(rec, status="input_required", status_message="Waiting for approval")
        answer = await rec["waiters"][key]
        self._set(rec, status="working", status_message="Approval received")
        return answer

    async def _run(self, rec, name, arguments):
        try:
            if self.task_tools[name].get("needs_approval"):
                answer = await self._ask(rec, "approve-1", {
                    "mode": "form", "message": f"Approve {name} with {arguments}?",
                    "requestedSchema": {"type": "object", "properties": {"approve": {"type": "boolean"}},
                                        "required": ["approve"]}})
                if answer.get("action") != "accept" or not (answer.get("content") or {}).get("approve"):
                    self._set(rec, status="completed", status_message="Declined by the user",
                              result={"content": [{"type": "text", "text": "Declined by the user."}],
                                      "isError": True})
                    return
            for step in (1, 2, 3):
                self._set(rec, status_message=f"step {step}/3")
                await asyncio.sleep(0.05)                       # pretend this is minutes of work
            out = await self.server.call_tool(name, arguments)  # run the real tool function
            self._set(rec, status="completed", status_message="Done", result=wire(out))
        except asyncio.CancelledError:
            self._set(rec, status="cancelled", status_message="Cancelled at the client's request")
        except MCPError as exc:                                 # a JSON-RPC error -> failed
            self._set(rec, status="failed", status_message=f"Tool execution failed: {exc.message}",
                      error=exc.error.model_dump(exclude_none=True))
        except Exception as exc:                                # a tool error -> completed + isError
            self._set(rec, status="completed", status_message="Tool reported an error",
                      result={"content": [{"type": "text", "text": str(exc)}], "isError": True})

    # ---- contribution points -------------------------------------------------------------------
    async def intercept_tool_call(self, params, ctx, call_next):
        policy = self.task_tools.get(params.name)
        caps = ctx.session.client_capabilities
        declared = bool(caps and caps.extensions and TASKS in caps.extensions)
        if policy is None:
            return await call_next(ctx)                         # not a long-running tool
        if not declared:
            if policy.get("requires_tasks"):
                require_client_extension(ctx, TASKS)            # cannot serve synchronously -> -32021
            return await call_next(ctx)                         # graceful degradation: block and answer
        task_id = secrets.token_urlsafe(16)                     # unguessable handle
        rec = {"task_id": task_id, "status": "working", "status_message": "Queued",
               "created_at": now(), "last_updated_at": now(), "ttl_ms": self.ttl_ms,
               "poll_interval_ms": self.poll_interval_ms, "requests": {}, "answers": {}, "waiters": {}}
        self.store[task_id] = rec                               # durably created BEFORE we respond
        rec["job"] = asyncio.create_task(self._run(rec, params.name, params.arguments or {}))
        return self._view(rec, CreateTaskResult)

    def methods(self):
        async def get(ctx, params: TaskIdParams):
            return self._view(self._lookup(ctx, params.task_id))

        async def update(ctx, params: UpdateTaskParams):
            rec = self._lookup(ctx, params.task_id)
            for key, response in params.input_responses.items():
                waiter = rec["waiters"].get(key)
                if waiter is None or key in rec["answers"]:     # unknown or already answered: ignore
                    continue
                rec["answers"][key] = response
                waiter.set_result(response)
            return types.EmptyResult(result_type="complete")

        async def cancel(ctx, params: TaskIdParams):
            rec = self._lookup(ctx, params.task_id)
            if rec["status"] not in TERMINAL:
                rec["job"].cancel()                             # cooperative: we choose to honour it
            return types.EmptyResult(result_type="complete")

        return [MethodBinding("tasks/get", TaskIdParams, get),
                MethodBinding("tasks/update", UpdateTaskParams, update),
                MethodBinding("tasks/cancel", TaskIdParams, cancel)]


tasks_ext = TasksExtension({"build_report": {"needs_approval": False, "requires_tasks": False},
                            "deploy": {"needs_approval": True, "requires_tasks": True}})
ops = MCPServer("ops", extensions=[tasks_ext])
tasks_ext.server = ops


@ops.tool()
def build_report(rows: int) -> str:
    """Aggregate `rows` rows of sales data into a report (slow)."""
    if rows > 1_000_000:
        raise MCPError(INTERNAL_ERROR, "Warehouse unavailable")   # protocol-level failure
    if rows < 0:
        raise ValueError("rows must be >= 0")                     # tool-level error
    return f"Report over {rows} rows: revenue up 12%."


@ops.tool()
def deploy(env: str) -> str:
    """Deploy the current build to an environment (needs human approval)."""
    return f"Deployed build 42 to {env}."


print("task-capable tools:", tasks_ext.task_tools)

### 7. 客户端：多态结果与轮询循环
声明了 Tasks 的客户端必须能接受<em>任意一种</em>结果形态。在 Python SDK 中，`ClientExtension` 通过一个以 `resultType` 为键的 `ResultClaim` 来声明额外的结果形态。借助 `session.call_tool(..., allow_claimed=True)`，我们拿到原始的 `CreateTaskResult`，然后自己驱动 `tasks/get`，并打印每一个线路结果。`name_param = "taskId"` 这一行会让 SDK 在通过 HTTP 运行时设置 `Mcp-Name: <taskId>`，从而支持对任务（Task）的路由。

In [ ]:
from mcp.client.extension import ClientExtension, ResultClaim


class GetTaskRequest(Request[TaskIdParams, Literal["tasks/get"]]):
    method: Literal["tasks/get"] = "tasks/get"
    name_param = "taskId"


class UpdateTaskRequest(Request[UpdateTaskParams, Literal["tasks/update"]]):
    method: Literal["tasks/update"] = "tasks/update"
    name_param = "taskId"


class CancelTaskRequest(Request[TaskIdParams, Literal["tasks/cancel"]]):
    method: Literal["tasks/cancel"] = "tasks/cancel"
    name_param = "taskId"


class TasksClient(ClientExtension):
    """Declares io.modelcontextprotocol/tasks and claims resultType "task"."""
    identifier = TASKS

    def __init__(self, answer=None):
        self.answer = answer   # callback used in section 9 to answer input requests

    def claims(self):
        return [ResultClaim(result_type="task", model=CreateTaskResult, resolve=self._resolve)]

    async def _resolve(self, created, cctx):
        raise NotImplementedError   # filled in section 9


async def tasks_get(session, task_id):
    return await session.send_request(GetTaskRequest(params=TaskIdParams(task_id=task_id)), DetailedTask)


async with Client(ops) as legacy, Client(ops, extensions=[TasksClient()]) as modern:
    print("server/discover extensions:", modern.server_capabilities.extensions)
    sync = await legacy.call_tool("build_report", {"rows": 500})          # no Tasks declared
    print("\nclient WITHOUT tasks -> ordinary CallToolResult:", sync.content[0].text)

    created = await modern.session.call_tool("build_report", {"rows": 500}, allow_claimed=True)
    print("\nclient WITH tasks -> tools/call result:")
    print(json.dumps(wire(created), indent=2))
    while True:
        await asyncio.sleep(created.poll_interval_ms / 1000)                # honour pollIntervalMs
        state = await tasks_get(modern.session, created.task_id)
        print("tasks/get ->", state.status, "|", state.status_message)
        if state.status in TERMINAL:
            print(json.dumps(wire(state)["result"], indent=2))
            break

### 8. 用 `tasks/update` 提供执行过程中的输入，然后看错误路径
`deploy` 需要审批。轮询直到 `input_required`，读取带键的 `inputRequests`，用 `tasks/update` 回答，再轮询直到完成。然后触发各种错误情况：未声明该扩展的客户端调用仅限任务（Task）的工具（`-32021`）、未知的 `taskId`（`-32602`）、一个 `failed` 的任务（JSON-RPC 错误），以及一个工具结果带有 `isError: true` 的 `completed` 任务。

In [ ]:
async def poll(session, task_id, interval=0.05):
    while True:
        state = await tasks_get(session, task_id)
        if state.status in TERMINAL or state.status == "input_required":
            return state
        await asyncio.sleep(interval)


async with Client(ops, extensions=[TasksClient()]) as c:
    created = await c.session.call_tool("deploy", {"env": "staging"}, allow_claimed=True)
    state = await poll(c.session, created.task_id)
    print("status:", state.status)
    print("inputRequests:", json.dumps(state.input_requests, indent=2))

    update = UpdateTaskRequest(params=UpdateTaskParams(
        task_id=created.task_id, input_responses={"approve-1": {"action": "accept", "content": {"approve": True}}}))
    print("\n>>> tasks/update params:", json.dumps(update.params.model_dump(by_alias=True, exclude_none=True)))
    ack = await c.session.send_request(update, types.EmptyResult)
    print("<<< ack:", wire(ack))
    final = await poll(c.session, created.task_id)
    print("final:", final.status, "->", final.result["content"][0]["text"])

    print("\n--- error paths ---")
    for rows in (2_000_000, -5):
        t = await c.session.call_tool("build_report", {"rows": rows}, allow_claimed=True)
        s = await poll(c.session, t.task_id)
        print(f"rows={rows:>9}: status={s.status:<9} error={s.error} isError={s.result and s.result.get('isError')}")
    try:
        await tasks_get(c.session, "no-such-task")
    except MCPError as err:
        print("unknown taskId ->", err.code, err.message)

async with Client(ops) as legacy:
    try:
        await legacy.call_tool("deploy", {"env": "prod"})
    except MCPError as err:
        print("deploy without Tasks ->", err.code, json.dumps(err.data))
    try:
        await tasks_get(legacy.session, "anything")
    except MCPError as err:
        print("tasks/get without Tasks ->", err.code)

### 9. 透明的客户端：隐藏任务的 `call_tool()`
规范中的向后兼容说明指出，客户端可以在内部驱动轮询循环，只返回最终的 `CallToolResult`。现在我们实现 `_resolve`，并传入一个 `answer` 回调，用它代替宿主（Host）的同意界面。已经回答过的键会在多次轮询之间去重。之后，`client.call_tool(...)` 就只是简单地返回结果。最后演示取消（Cancellation）。

In [ ]:
async def resolve(self, created, cctx):
    answered, interval = set(), created.poll_interval_ms or 1000
    while True:
        await asyncio.sleep(interval / 1000)
        state = await tasks_get(cctx.session, created.task_id)
        interval = state.poll_interval_ms or interval        # pollIntervalMs MAY change over time
        if state.status == "input_required":
            fresh = {k: v for k, v in (state.input_requests or {}).items() if k not in answered}
            if fresh:
                responses = {k: self.answer(v) for k, v in fresh.items()}
                answered |= set(fresh)
                await cctx.session.send_request(UpdateTaskRequest(params=UpdateTaskParams(
                    task_id=created.task_id, input_responses=responses)), types.EmptyResult)
        elif state.status == "completed":
            return types.CallToolResult.model_validate(state.result)
        elif state.status == "failed":
            raise MCPError(state.error["code"], state.error["message"])
        elif state.status == "cancelled":
            raise RuntimeError("task was cancelled")


TasksClient._resolve = resolve


def approve_everything(request):
    print("   [consent UI] ", request["params"]["message"], "-> approved")
    return {"action": "accept", "content": {"approve": True}}


async with Client(ops, extensions=[TasksClient(answer=approve_everything)]) as c:
    result = await c.call_tool("deploy", {"env": "production"})
    print("call_tool returned a plain CallToolResult:", result.content[0].text)

    created = await c.session.call_tool("build_report", {"rows": 10}, allow_claimed=True)
    await c.session.send_request(CancelTaskRequest(params=TaskIdParams(task_id=created.task_id)), types.EmptyResult)
    print("after tasks/cancel:", (await poll(c.session, created.task_id)).status)

### 动手练习
1. 在第 6 节中，让 `TasksExtension` 遵守 TTL。当 `createdAt + ttlMs` 已经过去时，`tasks/get` 应当返回 `-32602`，并附带 “Failed to retrieve task: Task has expired”。用 `ttl_ms=100` 进行测试。
2. 当某个任务被轮询超过五次后，让服务器把 `pollIntervalMs` 从 50 提高到 200。确认第 9 节的解析器会随之调整，因为它从最新状态中读取 `poll_interval_ms`。
3. 发送一个带有从未签发过的键的 `tasks/update`，再发送一个带有已经回答过的键的。检查服务器是否按规范要求，对两者都予以确认并忽略它们。
4. 让 `approve_everything` 改为拒绝。任务会进入哪个终止状态？为什么不是 `failed`？